In [1]:
pip install sentence-transformers pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

dem_path = '/Users/kimminseo/Documents/UCPH/ISP/dataset/reddit_democrats_clean.csv'
rep_path = '/Users/kimminseo/Documents/UCPH/ISP/dataset/reddit_republicans_clean.csv'

# sampling 5000 rows from each
dem_df = pd.read_csv(dem_path).sample(n=5000, random_state=42)
rep_df = pd.read_csv(rep_path).sample(n=5000, random_state=42)

dem_df['party'] = 'Democrat'
rep_df['party'] = 'Republican'

# combine data
combined_df = pd.concat([dem_df, rep_df], ignore_index=True)
print(f"data loaded")

# initialize the BERT Model 'all-mpnet-base-v2' map (768 dim)
print("\n downloading BERT model (all-mpnet-base-v2")
model = SentenceTransformer('all-mpnet-base-v2')


/Users/kimminseo/Documents/UCPH/ISP/pycode/ispenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/tq/pmn07sd93gs_hl4cdhsww3qh0000gn/T/ipykernel_60115/3596516478.py:9: DtypeWarning: Columns (8,9,10,16,17,21) have mixed types. Specify dtype option on import or set low_memory=False.
  dem_df = pd.read_csv(dem_path).sample(n=5000, random_state=42)
/var/folders/tq/pmn07sd93gs_hl4cdhsww3qh0000gn/T/ipykernel_60115/3596516478.py:10: DtypeWarning: Columns (8,9,10,17,21) have mixed types. Specify dtype option on import or set low_memory=False.
  rep_df = pd.read_csv(rep_path).sample(n=5000, random_state=42)


data loaded

 downloading BERT model (all-mpnet-base-v2


In [3]:
embeddings = model.encode(combined_df['self_text'].tolist(), show_progress_bar=True)
combined_df['bert_vector'] = list(embeddings)

print(f"Vector Shape: {embeddings.shape} (Rows, Dimensions)")
print("Example of one vector :")
print(combined_df['bert_vector'].iloc[0][:5])

Batches: 100%|██████████| 313/313 [01:20<00:00,  3.87it/s]

Vector Shape: (10000, 768) (Rows, Dimensions)
Example of one vector :
[-0.00198097  0.07034494  0.03533242  0.03316363 -0.01068578]


In [4]:
'''#anchor words

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


initial_seed = ["idiot", "stupid", "liar", "shame"]
seed_vec = model.encode(initial_seed)
seed_centroid = np.mean(seed_vec, axis=0).reshape(1, -1)

vectorizer = CountVectorizer(max_features=1000, stop_words='english')
vectorizer.fit(combined_df['self_text'])
vocab_words = list(vectorizer.vocabulary_.keys())

vocab_vectors = model.encode(vocab_words)

similarities = cosine_similarity(seed_centroid, vocab_vectors)[0]

top_indices = similarities.argsort()[-40:][::-1]
discovered_anchors = [vocab_words[i] for i in top_indices]

print("\n--- AGGRESSIVE ANCHORS ---")
print(discovered_anchors)


calm_seed = ["civil", "agree", "understand"]
calm_vec = model.encode(calm_seed)
calm_centroid = np.mean(calm_vec, axis=0).reshape(1, -1)

sim_calm = cosine_similarity(calm_centroid, vocab_vectors)[0]
top_calm_indices = sim_calm.argsort()[-40:][::-1]
discovered_calm = [vocab_words[i] for i in top_calm_indices]

print("\n--- CALM ANCHORS ---")
print(discovered_calm)'''

'#anchor words\n\nfrom sklearn.feature_extraction.text import CountVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\n\ninitial_seed = ["idiot", "stupid", "liar", "shame"]\nseed_vec = model.encode(initial_seed)\nseed_centroid = np.mean(seed_vec, axis=0).reshape(1, -1)\n\nvectorizer = CountVectorizer(max_features=1000, stop_words=\'english\')\nvectorizer.fit(combined_df[\'self_text\'])\nvocab_words = list(vectorizer.vocabulary_.keys())\n\nvocab_vectors = model.encode(vocab_words)\n\nsimilarities = cosine_similarity(seed_centroid, vocab_vectors)[0]\n\ntop_indices = similarities.argsort()[-40:][::-1]\ndiscovered_anchors = [vocab_words[i] for i in top_indices]\n\nprint("\n--- AGGRESSIVE ANCHORS ---")\nprint(discovered_anchors)\n\n\ncalm_seed = ["civil", "agree", "understand"]\ncalm_vec = model.encode(calm_seed)\ncalm_centroid = np.mean(calm_vec, axis=0).reshape(1, -1)\n\nsim_calm = cosine_similarity(calm_centroid, vocab_vectors)[0]\ntop_calm_indices = sim_calm.argsort()[

In [11]:
# looking for anchor sentences

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

# using the anchor words as seed words
seed_aggressive_words = [
    'stupid', 'dumb', 'bullshit', 'racist', 'lying', 'mad', 
    'nonsense', 'ridiculous', 'fraud', 'awful', 'fake', 
    'insane', 'shit', 'terrible', 'worst', 'lies', 'hypocrite'
]
seed_calm_words = [
    'understand', 'agree', 'civil', 'fair', 'reasonable', 
    'respect', 'reason', 'understanding', 'fine', 'opinion', 
    'sense', 'okay', 'discussion', 'consider'
]

seed_agg_vec = model.encode(seed_aggressive_words)
agg_centroid = np.mean(seed_agg_vec, axis=0).reshape(1, -1)

seed_calm_vec = model.encode(seed_calm_words)
calm_centroid = np.mean(seed_calm_vec, axis=0).reshape(1, -1)

all_vectors = np.stack(combined_df['bert_vector'].values)

sim_agg = cosine_similarity(all_vectors, agg_centroid).flatten()
sim_calm = cosine_similarity(all_vectors, calm_centroid).flatten()

def get_best_anchors(similarity_scores, df, n_anchors=10):
    temp_df = df.copy()
    temp_df['score'] = similarity_scores
    
    candidates = temp_df[
        (temp_df['self_text'].str.len() >= 30) & 
        (temp_df['self_text'].str.len() <= 300)
    ].sort_values('score', ascending=False).head(100)

    candidate_vecs = np.stack(candidates['bert_vector'].values)
    kmeans = KMeans(n_clusters=n_anchors, random_state=42)
    kmeans.fit(candidate_vecs)
    
    anchors = []
    for i in range(n_anchors):
        center = kmeans.cluster_centers_[i].reshape(1, -1)

        dists = cosine_similarity(candidate_vecs, center).flatten()
        best_idx = dists.argmax()
        anchors.append(candidates.iloc[best_idx]['self_text'])
        
    return anchors

discovered_aggressive_anchors = get_best_anchors(sim_agg, combined_df, n_anchors=10)
discovered_calm_anchors = get_best_anchors(sim_calm, combined_df, n_anchors=10)

print("\n--- AGGRESSIVE ANCHORS ---")
for s in discovered_aggressive_anchors: print(f"- {s}")

print("\n--- CALM ANCHORS ---")
for s in discovered_calm_anchors: print(f"- {s}")


--- AGGRESSIVE ANCHORS ---
- Probably more of a Russian troll.
- Nobody is as dumb as you think they are.
- I’m mad I ever supported this piece of shit fucking sellout
- Sorry that is disagreeable. Unjustified what they’ve done to him any way you sniff it.
- What a fucking insane thing to say. 

Hard to grasp.
- Republicans outrage over [fill in the blank]
- A pure maga moron spewing hate, disinformation and lies
- FAKE NUMBERS. FAKE NEWS!!!11!!!!!11!!!!!  
/S
- LITERALLY!! THIS IS STILL AWFUL
- He lied on voter forms multiple times.

--- CALM ANCHORS ---
- Not sure you made a point of discussion at all
- I’m not a fan of reason but they are pretty consistent with their ideology
- i agree - lets take a poll on the question
- True. Asking for what specific aspect of the verdict, brought unanimously by 12 of their peers, they object to, is usually (not always) met with silence.
- This is not going anywhere useful.
- Strength in numbers. We must support each other publicly and privately.

In [17]:
def get_centroid(vectors):
    return np.mean(np.vstack(vectors), axis=0)

def create_axis(positive_vectors, negative_vectors):
    pos_center = get_centroid(positive_vectors)
    neg_center = get_centroid(negative_vectors)
    return pos_center - neg_center

def calculate_score(vector, axis):
    v = vector.reshape(1, -1)
    ax = axis.reshape(1, -1)
    return cosine_similarity(v, ax)[0][0]

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_all = np.vstack(combined_df['bert_vector'].values)
print(f"Feature Matrix Shape: {X_all.shape}")

# A. Polarization

print("\nTraining Polarization Classifier...")

y_party = combined_df['party']
X_train_pol, X_test_pol, y_train_pol, y_test_pol = train_test_split(
    X_all, y_party, test_size=0.2, random_state=42, stratify=y_party
)

pol_model = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs')
pol_model.fit(X_train_pol, y_train_pol)

print("Polarization Validation Report:")
print(classification_report(y_test_pol, pol_model.predict(X_test_pol)))

combined_df['model_polarization_score'] = pol_model.predict_proba(X_all)[:, 1]

print("Creating High-Signal Polarization Axis...")
dem_high_sig = combined_df[(combined_df['party']=='Democrat') & (combined_df['self_text'].str.len() > 50)]
rep_high_sig = combined_df[(combined_df['party']=='Republican') & (combined_df['self_text'].str.len() > 50)]

dem_vectors = dem_high_sig.sort_values('self_text', key=lambda x: x.str.len(), ascending=False).head(1000)['bert_vector'].tolist()
rep_vectors = rep_high_sig.sort_values('self_text', key=lambda x: x.str.len(), ascending=False).head(1000)['bert_vector'].tolist()

pol_axis_heuristic = create_axis(rep_vectors, dem_vectors)
combined_df['heuristic_pol_score'] = combined_df['bert_vector'].apply(lambda x: calculate_score(x, pol_axis_heuristic))

Feature Matrix Shape: (10000, 768)

Training Polarization Classifier...
Polarization Validation Report:
              precision    recall  f1-score   support

    Democrat       0.60      0.60      0.60      1000
  Republican       0.60      0.59      0.60      1000

    accuracy                           0.60      2000
   macro avg       0.60      0.60      0.60      2000
weighted avg       0.60      0.60      0.60      2000

Creating High-Signal Polarization Axis...


In [21]:
# B. Agressiveness

print("\nTraining Aggressiveness Classifier...")

agg_vectors = model.encode(discovered_aggressive_anchors)
calm_vectors = model.encode(discovered_calm_anchors)
axis_agg_heuristic = create_axis(agg_vectors, calm_vectors)

combined_df['agg_score'] = combined_df['bert_vector'].apply(
    lambda x: calculate_score(x, axis_agg_heuristic)
)

df_sorted = combined_df.sort_values('agg_score')
n_samples = int(len(df_sorted) * 0.10)

calm_train = df_sorted.head(n_samples).copy()
agg_train  = df_sorted.tail(n_samples).copy()

calm_train['training_label'] = 0
agg_train['training_label']  = 1

train_df = pd.concat([calm_train, agg_train])
X_train_agg = np.vstack(train_df['bert_vector'].values)
y_train_agg = train_df['training_label']

agg_model = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs')
agg_model.fit(X_train_agg, y_train_agg)

combined_df['model_aggressiveness_score'] = agg_model.predict_proba(X_all)[:, 1]


Training Aggressiveness Classifier...


In [ ]:
# validation

cols_to_check = [
    'self_text', 
    'model_polarization_score',
    'agg_score',
    'model_aggressiveness_score'
]

print(combined_df[cols_to_check].head())

top_agg = combined_df.sort_values('model_aggressiveness_score', ascending=False).head(5)
top_calm = combined_df.sort_values('model_aggressiveness_score', ascending=True).head(5)

print("\n>>> Model: Most Aggressive:")
for i, row in top_agg.iterrows():
    print(f"[Score: {row['model_aggressiveness_score']:.4f}] {row['self_text'][:100]}...")

print("\n>>> Model: Calmest:")
for i, row in top_calm.iterrows():
    print(f"[Score: {row['model_aggressiveness_score']:.4f}] {row['self_text'][:100]}...")

                                           self_text  \
0  Reminds me of the guy I saw 20+ years ago who ...   
1  In my experience if you want things done in th...   
2  A messaging issue can place blame on the idiot...   
3  This entire thing was a Kompromat Op to contro...   
4  You seem to have missed Bidens great speech to...   

   model_polarization_score  agg_score  model_aggressiveness_score  
0                  0.691047   0.089606                    0.683673  
1                  0.328405  -0.074658                    0.027732  
2                  0.252837  -0.070953                    0.156566  
3                  0.609558   0.141119                    0.892689  
4                  0.398027   0.176060                    0.979032  

>>> Model: Most Aggressive:
[Score: 0.9985] This after bragging about the benefit of his campaign as being this  populist billionaire not behold...
[Score: 0.9978] Here is one of Trump's bootlicker now. The ones in the media who live in Trump's ass

In [14]:
print("\nAggressiveness Axis (manually defined)")
agg_vectors = model.encode(discovered_aggressive_anchors)
calm_vectors = model.encode(discovered_calm_anchors)
aggressiveness_axis = create_axis(agg_vectors, calm_vectors)

print("Creating Polarization Axis")
dem_vectors = combined_df[combined_df['party'] == 'Democrat']['bert_vector'].tolist()
rep_vectors = combined_df[combined_df['party'] == 'Republican']['bert_vector'].tolist()
polarization_axis = create_axis(rep_vectors, dem_vectors)

print("\nCalculating final scores")

combined_df['score_polarization'] = combined_df['bert_vector'].apply(
    lambda x: calculate_score(x, polarization_axis)
)
combined_df['score_aggressiveness'] = combined_df['bert_vector'].apply(
    lambda x: calculate_score(x, aggressiveness_axis)
)


Aggressiveness Axis (manually defined)
Creating Polarization Axis

Calculating final scores


In [16]:
# Agressiveness Model Validation

top_agg_model = combined_df.sort_values('model_aggressiveness_score', ascending=False).head(5)
top_calm_model = combined_df.sort_values('model_aggressiveness_score', ascending=True).head(5)

print("\n>>> Most Agressive (Prob ~ 1.0):")
for i, row in top_agg_model.iterrows():
    print(f"[Model Prob: {row['model_aggressiveness_score']:.4f}] {row['self_text'][:100]}...")

print("\n>>>Calmest (Prob ~ 0.0):")
for i, row in top_calm_model.iterrows():
    print(f"[Model Prob: {row['model_aggressiveness_score']:.4f}] {row['self_text'][:100]}...")


>>> Most Agressive (Prob ~ 1.0):
[Model Prob: 0.9985] This after bragging about the benefit of his campaign as being this  populist billionaire not behold...
[Model Prob: 0.9978] Here is one of Trump's bootlicker now. The ones in the media who live in Trump's ass.

I remember wh...
[Model Prob: 0.9978] It's beyond appalling that anyone can see this decrepit child losing his shit and unable to control ...
[Model Prob: 0.9976] He's a crooked creep that stands for nothing except doing whatever he's told. He's hateful, vindicti...
[Model Prob: 0.9975] The background of this story is also disgusting.  He sold out a Presidential Medal of Honor to Shel ...

>>>Calmest (Prob ~ 0.0):
[Model Prob: 0.0009] I'm not saying that they are needed all the time everywhere, I'm saying it's logical to have one...
[Model Prob: 0.0012] I agree. I just don't know how one would go about that....
[Model Prob: 0.0015] you are correct. this is probably how the idea should be pitched but it just doesn't roll off